# Sales Performance Investigation

## Business Scenario

Management has raised concerns about sales performance and wants to understand which salespeople and regions are underperforming, what patterns may be contributing to the weaker results, and what additional information is needed before making business decisions.

In this project, I use **MySQL** and SQL analysis to investigate sales performance across salespeople and regions.

### Questions investigated

- What are the overall sales, orders, returns, and average sales per order?
- Which salesperson generates the most and least sales?
- Which region performs better?
- Does the regional performance gap remain after accounting for the number of salespeople?
- Are differences in sales driven by order volume or order value?
- Which salesperson has the highest returns and discount levels?
- What additional data is needed to determine the underlying cause of weak performance?


## `sales` table

The analysis uses the following columns:

| Column | Definition | Data Type |
|---|---|---|
| `salesperson` | Name of the salesperson | VARCHAR |
| `region` | Region assigned to the salesperson | VARCHAR |
| `sales` | Sales value | DECIMAL / NUMERIC |
| `discount` | Discount associated with the sale | DECIMAL / NUMERIC |
| `orders` | Number of orders | INT |
| `returns` | Number of returned orders | INT |


## 1. Overall Sales Performance

Before investigating the cause of weak performance, establish the overall sales position.

In [ ]:
SELECT
    SUM(sales) AS total_sales,
    SUM(orders) AS total_orders,
    SUM(returns) AS total_returns,
    ROUND(SUM(sales) / SUM(orders), 2) AS avg_sales_per_order
FROM sales;

### Interpretation

The overall metrics provide the baseline for comparing individual salespeople and regions. The average sales per order is calculated as total sales divided by total orders.

## 2. Sales Performance by Salesperson

Which salesperson contributes the most and least sales?

In [ ]:
SELECT
    salesperson,
    SUM(sales) AS total_sales,
    SUM(orders) AS total_orders
FROM sales
GROUP BY salesperson
ORDER BY total_sales DESC;

### Observation

Carol has the highest sales and the highest number of orders. Brian has the lowest sales and the lowest number of orders, making him the main salesperson requiring further investigation.

## 3. Average Sales per Order by Salesperson

The next question is whether lower sales are explained only by fewer orders, or whether each order is also generating less revenue.

In [ ]:
SELECT
    salesperson,
    total_sales,
    total_orders,
    ROUND(total_sales / total_orders, 2) AS avg_sales_per_order
FROM (
    SELECT
        salesperson,
        SUM(sales) AS total_sales,
        SUM(orders) AS total_orders
    FROM sales
    GROUP BY salesperson
) AS salesperson_sales
ORDER BY total_sales DESC;

### Observation

Carol has the highest total sales, but she does not have the highest average sales per order. Brian has the lowest average sales per order. This indicates that Brian's weaker performance is not only a volume issue; his orders also generate less revenue on average.

## 4. Returns and Discounts by Salesperson

Investigate whether Brian's weaker performance is accompanied by unusually high returns or discounting.

In [ ]:
SELECT
    salesperson,
    SUM(sales) AS total_sales,
    SUM(orders) AS total_orders,
    SUM(returns) AS total_returns,
    AVG(discount) AS avg_discount
FROM sales
GROUP BY salesperson
ORDER BY total_sales DESC;

### Observation

Brian has the highest average discount at 11.0, compared with 6.6 for Alice and 5.5 for Carol. This is an important pattern, but the dataset does not prove that higher discounting causes Brian's weaker performance.

## 5. Return Rate by Salesperson

A high number of returns may reduce effective sales performance, so calculate returns as a percentage of orders.

In [ ]:
SELECT
    salesperson,
    SUM(returns) AS total_returns,
    SUM(orders) AS total_orders,
    ROUND(SUM(returns) / SUM(orders) * 100, 2) AS return_rate
FROM sales
GROUP BY salesperson
ORDER BY return_rate DESC;

### Observation

Brian has a return rate of 33.33%, compared with 15.79% for Alice and 8.70% for Carol. The difference is large enough to warrant investigation. However, return rate alone does not establish the cause of the weaker sales performance.

## 6. Regional Performance

Compare sales, orders, returns, discounting, and average sales per order by region.

In [ ]:
SELECT
    region,
    total_sales,
    total_orders,
    total_returns,
    avg_discount,
    ROUND(total_sales / total_orders, 2) AS avg_sales_per_order
FROM (
    SELECT
        region,
        SUM(sales) AS total_sales,
        SUM(orders) AS total_orders,
        SUM(returns) AS total_returns,
        AVG(discount) AS avg_discount
    FROM sales
    GROUP BY region
) AS regional_sales;

### Observation

Western generates KSh 215,000 in sales compared with KSh 65,000 in Eastern. Eastern also has the higher return rate and average discount. Western has the higher average sales per order at approximately KSh 5,119.05.

## 7. Normalize Regional Performance by Salesperson Count

Western has two salespeople while Eastern has one. Comparing raw regional totals alone could therefore be misleading.

In [ ]:
SELECT
    region,
    total_sales,
    number_of_salesperson,
    ROUND(total_sales / number_of_salesperson, 2) AS sales_per_person
FROM (
    SELECT
        region,
        SUM(sales) AS total_sales,
        COUNT(DISTINCT salesperson) AS number_of_salesperson
    FROM sales
    GROUP BY region
) AS sales_per_salesperson_per_region;

### Result

| Region | Sales | Salespeople | Sales per Person |
|---|---:|---:|---:|
| Eastern | KSh 65,000 | 1 | KSh 65,000 |
| Western | KSh 215,000 | 2 | KSh 107,500 |

Eastern remains weaker even after accounting for the different number of salespeople.

## 8. Orders per Salesperson by Region

Determine whether Eastern's weaker sales are also associated with lower order productivity.

In [ ]:
SELECT
    region,
    total_orders,
    number_of_salesperson,
    ROUND(total_orders / number_of_salesperson, 2) AS orders_per_person
FROM (
    SELECT
        region,
        SUM(orders) AS total_orders,
        COUNT(DISTINCT salesperson) AS number_of_salesperson
    FROM sales
    GROUP BY region
) AS sales_per_salesperson_per_region;

### Result

| Region | Orders | Salespeople | Orders per Person |
|---|---:|---:|---:|
| Eastern | 15 | 1 | 15.00 |
| Western | 42 | 2 | 21.00 |

Eastern generates fewer orders per salesperson than Western.

## 9. Average Sales per Order by Region

The final comparison separates the sales problem into two components: order volume and order value.

In [ ]:
SELECT
    region,
    total_sales,
    total_orders,
    ROUND(total_sales / total_orders, 2) AS avg_sales_per_order
FROM (
    SELECT
        region,
        SUM(sales) AS total_sales,
        SUM(orders) AS total_orders
    FROM sales
    GROUP BY region
) AS regional_sales_per_order;

### Result

| Region | Sales | Orders | Average Sales per Order |
|---|---:|---:|---:|
| Eastern | KSh 65,000 | 15 | ~KSh 4,333.33 |
| Western | KSh 215,000 | 42 | ~KSh 5,119.05 |

Eastern's weaker sales are associated with both lower order volume and lower average revenue per order.

## Key Findings

1. **Brian is the primary salesperson requiring investigation.** He has the lowest sales, fewest orders, lowest average sales per order, highest average discount, and highest return rate.

2. **Eastern is the weaker region.** Even after adjusting for salesperson count, Eastern generates KSh 65,000 per salesperson compared with KSh 107,500 in Western.

3. **Eastern's weaker performance is not explained by salesperson count alone.** Eastern has fewer orders per salesperson (15 vs. 21) and lower average sales per order (~KSh 4,333 vs. ~KSh 5,119).

4. **High discounting and returns are important warning signals.** Brian's 11% average discount and 33.33% return rate deserve further investigation, but the current dataset cannot prove causation.


## Root-Cause Hypotheses

The current dataset identifies where the performance problem appears, but it does not establish why it is happening.

Potential factors to investigate include:

- Sales strategy and sales channels
- Product mix and product prices
- Customer segments and customer acquisition
- Customer service and satisfaction
- Reasons for returned products
- Product handling and delivery quality
- Seasonal or time-based sales patterns


## Recommendations

### Immediate

- Investigate Brian's unusually high return rate and discounting.
- Review sales activity and order-generation methods in Eastern.
- Compare the products and customer types served in Eastern with those in Western.

### Further analysis

Management should collect or provide additional data on product category, customer segment, sales channel, return reason, customer satisfaction, and sales date/month. This would allow the business to distinguish between salesperson performance and regional or market factors.

### Important limitation

Brian is the only salesperson in Eastern. Therefore, this dataset cannot cleanly separate the effect of the salesperson from the effect of the Eastern market. The findings should be treated as evidence for further investigation rather than proof of a single root cause.

## Conclusion

The SQL investigation identifies Brian and the Eastern region as the main areas requiring attention. Eastern underperforms Western not only in total sales but also in sales per salesperson, orders per salesperson, and average sales per order. Brian also shows unusually high discounting and returns.

However, the available data is insufficient to establish the underlying cause. A stronger business decision would require product, customer, sales-channel, return-reason, customer-service, and time-based data.

**Tools:** MySQL, SQL, GitHub/Jupyter Notebook
